[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C76_Causal_Inference_Course/01_potential_outcomes/01_potential_outcomes.ipynb)

# C76 模块 01 · 潜在结果框架

三条主线：

1. **四个目标量**（ATE / ATT / ATC / CATE）数值上不相等，先选再估；
2. **识别 vs 估计**：分层估计在可忽略性成立时收敛，未观测混杂下偏差不随 $n$ 消失；
3. **CATE 定向**的收益由异质性强度决定，异质性弱时定向是净亏。

纯 numpy / CPU / 离线。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def make(n=200_000, seed=0, uconf=0.0, h=0.75):
    '''构造潜在结果全部已知的人群。

    x     : 可观测协变量
    U     : **未观测**混杂（uconf=0 时不起作用）
    tau   : 个体效应 = 1 - h*x（h 控制异质性强度）
    T     : 处理概率随 x 和 U 递增
    '''
    r = np.random.default_rng(seed)
    x  = r.normal(0.0, 1.0, n)
    U  = r.normal(0.0, 1.0, n)
    Y0 = 1.0 * x + uconf * U + r.normal(0.0, 0.5, n)
    tau = 1.0 - h * x
    Y1 = Y0 + tau
    p  = 1.0 / (1.0 + np.exp(-(1.2 * x + uconf * U)))
    T  = (r.random(n) < p).astype(float)
    Y  = np.where(T == 1, Y1, Y0)
    return dict(x=x, U=U, Y0=Y0, Y1=Y1, tau=tau, T=T, Y=Y, p=p)

d = make()
print(f"n = {len(d['T']):,}，处理率 = {d['T'].mean():.4f}")
print()
print('四个目标量：')
print(f"  ATE  = {d['tau'].mean():+.6f}")
print(f"  ATT  = {d['tau'][d['T']==1].mean():+.6f}")
print(f"  ATC  = {d['tau'][d['T']==0].mean():+.6f}")
print(f"  ATT/ATC = {d['tau'][d['T']==1].mean()/d['tau'][d['T']==0].mean():.3f} 倍")
print()
print('CATE 是一个函数，不是一个数：')
for xv in (-2.0, -1.0, 0.0, 1.0, 2.0):
    print(f'  CATE(x={xv:+.1f}) = {1.0 - 0.75*xv:+.4f}')

## 1. 全期望公式：三个常数量的线性关系

$$\text{ATE} = \pi\,\text{ATT} + (1-\pi)\,\text{ATC}, \qquad \pi = P(T{=}1)$$

In [ ]:
att = d['tau'][d['T'] == 1].mean()
atc = d['tau'][d['T'] == 0].mean()
pi  = d['T'].mean()
ate = d['tau'].mean()

print(f'{pi:.6f} * {att:.6f} + {1-pi:.6f} * {atc:.6f} = {pi*att + (1-pi)*atc:.6f}')
print(f'真 ATE                                            = {ate:.6f}')
print(f'差                                                = {abs(pi*att+(1-pi)*atc - ate):.3e}')
assert abs(pi*att + (1-pi)*atc - ate) < 1e-12

print()
print('ATT - ATC 的来源是 Cov(tau, T)：')
cov = np.cov(d['tau'], d['T'], ddof=0)[0, 1]
print(f'  Cov(tau, T)              = {cov:+.6f}')
print(f'  Cov/(pi*(1-pi))          = {cov/(pi*(1-pi)):+.6f}')
print(f'  ATT - ATC                = {att-atc:+.6f}')
assert abs(cov/(pi*(1-pi)) - (att-atc)) < 1e-9
print()
print('-> tau 与 T 不相关时三者相等；相关时必须先说清报告哪一个。')

## 2. 分层估计：可忽略性成立时它收敛

把 $X$ 按分位数切成 $K$ 层，层内做差，按层大小加权。
这是最朴素的、也是最透明的后门调整估计量。

In [ ]:
def strat_ate(x, T, Y, K, return_dropped=False):
    '''分层（subclassification）估计 ATE。

    参数
    ----
    x : 分层用的协变量 (n,)
    T : 处理 (n,)
    Y : 观测结果 (n,)
    K : 层数
    return_dropped : 是否额外返回被丢弃样本的比例

    返回
    ----
    float : 按层大小加权的层内差
    （return_dropped=True 时返回 (估计, 丢弃比例)）
    '''
    qs = np.quantile(x, np.linspace(0, 1, K + 1))
    qs[0] -= 1e-9
    qs[-1] += 1e-9
    b = np.digitize(x, qs[1:-1])
    num, den, dropped = 0.0, 0, 0
    for k in range(K):
        m = (b == k)
        if m.sum() == 0:
            continue
        t1, t0 = m & (T == 1), m & (T == 0)
        if t1.sum() == 0 or t0.sum() == 0:
            dropped += int(m.sum())       # 该层缺一臂 -> 无法比较，静默跳过
            continue
        num += m.sum() * (Y[t1].mean() - Y[t0].mean())
        den += m.sum()
    if return_dropped:
        return float(num / den), dropped / len(x)
    return float(num / den)

naive = d['Y'][d['T'] == 1].mean() - d['Y'][d['T'] == 0].mean()
b1 = naive - ate
print(f'真 ATE = {ate:.6f}，朴素差 = {naive:.6f}（偏差 {b1:+.6f}）')
print()
print('  层数 K    分层估计      偏差         消除的偏差比例')
for K in (1, 2, 3, 5, 8, 10, 20, 50, 200):
    e = strat_ate(d['x'], d['T'], d['Y'], K)
    frac = (1 - abs(e - ate) / abs(b1)) * 100
    tag = '  <- Cochran 的五层规则' if K == 5 else ('  <- 反弹（层内样本不足）' if K == 200 else '')
    print(f'  {K:6d}   {e:+.6f}   {e-ate:+.6f}      {frac:6.1f}%{tag}')

_e5 = strat_ate(d['x'], d['T'], d['Y'], 5)
assert 0.85 < (1 - abs(_e5-ate)/abs(b1)) < 0.95, 'K=5 应消除约 90% 的偏差'
_e50 = strat_ate(d['x'], d['T'], d['Y'], 50)
_e200 = strat_ate(d['x'], d['T'], d['Y'], 200)
assert abs(_e200-ate) > abs(_e50-ate), 'K=200 的偏差应比 K=50 更大（反弹）'
print()
print('-> 分层也有偏差-方差权衡：K 太小残余混杂，K 太大层内样本不足。')

## 3. 识别失败：偏差不随 $n$ 消失

加入一个未观测混杂 $U$，然后**仍然控制全部可观测变量**（$K{=}20$ 分层）。

In [ ]:
print('  情形            n=2e3      n=2e4      n=2e5      n=2e6      走向')
rows = {}
for tag, uc in [('u=0（无混杂）', 0.0), ('u=1（有混杂）', 1.0), ('u=2（强混杂）', 2.0)]:
    row = []
    for n in (2_000, 20_000, 200_000, 2_000_000):
        dd = make(n=n, seed=1, uconf=uc)
        row.append(strat_ate(dd['x'], dd['T'], dd['Y'], 20) - dd['tau'].mean())
    rows[uc] = row
    trend = '-> 0' if abs(row[-1]) < 0.02 else f'-> {row[-1]:+.3f} 不收敛'
    print(f'  {tag:14s} ' + '  '.join(f'{v:+.4f}' for v in row) + f'   {trend}')

assert abs(rows[0.0][-1]) < 0.02, '无混杂时偏差应趋于 0'
assert abs(rows[1.0][-1]) > 0.5, '有混杂时偏差不应消失'
assert abs(rows[2.0][-1]) > abs(rows[1.0][-1]), '混杂越强偏差越大'

print()
print(f'样本量放大 1000 倍，u=1 的偏差从 {rows[1.0][0]:+.4f} 变到 {rows[1.0][-1]:+.4f}')
print('-> 一位有效数字都没动。这是识别问题，不是估计问题。')
print('   症状：置信区间越来越窄，而窄区间的中心是错的。')

## 4. CATE 定向：收益由异质性强度决定

In [ ]:
def targeting_gain(h, noise_sd, n=400_000, seed=2):
    '''按带噪声的 CATE 估计决定是否投放，返回相对全量投放的收益倍数。'''
    r = np.random.default_rng(seed)
    x = r.normal(0, 1, n)
    tau = 1.0 - h * x
    base = tau.mean()                                  # 全量投放的人均收益
    rn = np.random.default_rng(9)
    tau_hat = tau + rn.normal(0, noise_sd, n)           # CATE 估计（带噪声）
    gain = tau[tau_hat > 0].sum() / n                   # 只对估计为正的人投放
    return float(gain / base), float(base), float(tau.std()), float((tau < 0).mean())

print('   h    sd(tau)  负效应占比   ATE      sd=0.25  sd=0.5   sd=1.0   sd=2.0')
for h in (0.25, 0.75, 1.5, 3.0, 6.0):
    vals = [targeting_gain(h, sd)[0] for sd in (0.25, 0.5, 1.0, 2.0)]
    _, base, sdt, negf = targeting_gain(h, 0.25)
    flag = '   <- 每档都净亏' if all(v < 1.0 for v in vals) else ''
    print(f'  {h:4.2f}   {sdt:6.3f}   {negf*100:6.1f}%   {base:.4f}   '
          + '  '.join(f'{v:6.3f}' for v in vals) + flag)

_v = [targeting_gain(0.25, sd)[0] for sd in (0.25, 0.5, 1.0, 2.0)]
assert all(v < 1.0 for v in _v), 'h=0.25（无异质性）时定向应全部净亏'
_v6 = [targeting_gain(6.0, sd)[0] for sd in (0.25, 0.5, 1.0, 2.0)]
assert all(v > 2.5 for v in _v6), 'h=6.0 时定向应有数倍收益'

print()
print('三点：')
print('  1. ATE 在这五行里几乎不变（0.988-0.999），所以看 ATE 判断不了定向有没有价值。')
print('  2. h=0.25 时没有可用信号，噪声只制造错误排除 -> 每档都亏，最差 0.702 倍。')
print('  3. 盈亏平衡的噪声水平 sd* ∝ sd(tau)^2 / ATE（练习 4 会把这条律测出来）。')

## ✏️ 练习 1：ATT 的「反事实」写法

ATT 可以写成

$$\text{ATT} = E[Y \mid T{=}1] - E[Y(0) \mid T{=}1]$$

第二项是**处理组的反事实均值**，真实数据里不可见。
实现 `att_from_counterfactual(Y, T, Y0_treated_mean)`。

In [ ]:
def att_from_counterfactual(Y, T, Y0_treated_mean):
    '''由处理组的反事实均值算 ATT。

    参数
    ----
    Y                : 观测结果 (n,)
    T                : 处理 (n,)
    Y0_treated_mean  : E[Y(0) | T=1]，处理组若未被处理的均值

    返回
    ----
    float : ATT
    '''
    # TODO: ATT = E[Y|T=1] - E[Y(0)|T=1]
    raise NotImplementedError

In [ ]:
# 自测
_d = make(n=80_000, seed=4)
_T, _Y, _Y0, _tau = _d['T'], _d['Y'], _d['Y0'], _d['tau']
_cf = _Y0[_T == 1].mean()                       # 上帝视角
_true_att = _tau[_T == 1].mean()

_got = att_from_counterfactual(_Y, _T, _cf)
assert abs(_got - _true_att) < 1e-12, f'应精确等于 ATT：{_got} vs {_true_att}'

# 若错误地用对照组均值代替反事实均值，就退化成朴素差
_wrong = att_from_counterfactual(_Y, _T, _Y0[_T == 0].mean())
_naive = _Y[_T == 1].mean() - _Y[_T == 0].mean()
assert abs(_wrong - _naive) < 1e-12, '用对照组均值代替反事实均值应得到朴素差'

print(f'✅ 用真实反事实均值 {_cf:+.6f} -> ATT = {_got:+.6f}（真值 {_true_att:+.6f}）')
print(f'✅ 误用对照组均值 {_Y0[_T==0].mean():+.6f} -> {_wrong:+.6f} = 朴素差 {_naive:+.6f}')
print(f'   两者相差 {abs(_got-_wrong):.6f} = 选择偏差')

## ✏️ 练习 2：分层数的偏差-方差权衡

实现 `strat_rmse(K, seeds)`：在固定 $n$ 下，对多个 seed 跑 $K$ 层分层估计，
返回相对真 ATE 的 RMSE。用它找出 RMSE 最小的 $K$。

In [ ]:
def strat_rmse(K, seeds=20, n=4_000):
    '''K 层分层估计的 RMSE 与被丢弃样本比例（跨 seed）。

    参数
    ----
    K     : 层数
    seeds : 重复次数
    n     : 每次的样本量（故意取小，让方差可见）

    返回
    ----
    (float, float) : (sqrt(mean((估计 - 真 ATE)^2)), 平均被丢弃样本比例)
    '''
    # TODO: 对 seed in range(seeds)，用 make(n=n, seed=100+seed) 生成数据，
    #       用 strat_ate(..., K, return_dropped=True) 估计，
    #       与该次的 tau.mean() 比较，返回 (RMSE, 平均丢弃比例)
    raise NotImplementedError

In [ ]:
# 自测
_res, _drop = {}, {}
print('   K     RMSE      被丢弃的样本')
for _K in (1, 2, 5, 10, 25, 50, 100, 200, 400, 800):
    _res[_K], _drop[_K] = strat_rmse(_K)
    print(f'  {_K:4d}   {_res[_K]:.5f}      {_drop[_K]*100:5.1f}%')

_best = min(_res, key=_res.get)
assert _res[1] > _res[_best], 'K=1（朴素差）不该是最优'
assert 5 <= _best <= 400, f'最优层数应在中间，得到 {_best}'
assert _res[1] > 0.3, f'n=4000 时朴素差的 RMSE 应由偏差主导（>0.3），得到 {_res[1]:.4f}'
assert _drop[50] == 0.0, 'K=50 时不该有层缺臂'
assert _drop[400] > 0.05, f'K=400 应丢弃 >5% 的样本，实测 {_drop[400]*100:.1f}%'
assert _drop[800] > 0.15, f'K=800 应丢弃 >15% 的样本，实测 {_drop[800]*100:.1f}%'

print()
print(f'✅ RMSE 最小的层数 K* = {_best}（RMSE {_res[_best]:.5f}）')
print(f'   K=1 的 RMSE {_res[1]:.5f} 由**偏差**主导（n=4000 下偏差 ~0.58）')
print(f'   而 RMSE 选出的 K*={_best} 已经丢弃了 {_drop[_best]*100:.1f}% 的样本 ——')
print('   所以 RMSE 本身不是选层数的正确判据（它相对的是一个已被换掉的目标）。')
print()
print('⚠️  注意 K=400 的 RMSE 反而比 K=100 更**低**：')
print(f'   K=100 RMSE {_res[100]:.5f}（丢弃 {_drop[100]*100:.1f}%）')
print(f'   K=400 RMSE {_res[400]:.5f}（丢弃 {_drop[400]*100:.1f}%）')
print(f'   K=800 RMSE {_res[800]:.5f}（丢弃 {_drop[800]*100:.1f}%）')
print('   原因不是估计变好了，而是缺一臂的层被**静默跳过**：')
print('   估计目标从「全人群 ATE」悄悄变成了「两臂齐全的层上的 ATE」。')
print('   RMSE 之所以还低，是因为它是相对**原来那个真值**算的，而目标已经换了。')
print('   -> 这是本课反复出现的模式：失效不报错，只换掉你在估的东西。')

## ✏️ 练习 3：未观测混杂的敏感性

给定一个未观测混杂强度 $u$，实现 `residual_bias(u, n)` 返回
「控制全部可观测变量后」的残余偏差。用它画出偏差随 $u$ 的曲线。

这是**敏感性分析**（sensitivity analysis）的最简形式：
既然不能检验可忽略性，就问「$U$ 要多强，才能把结论翻过来」。

In [ ]:
def residual_bias(u, n=100_000, K=20, seed=1):
    '''控制全部可观测变量后的残余偏差。

    参数
    ----
    u : 未观测混杂强度（make 的 uconf 参数）
    n : 样本量
    K : 分层数

    返回
    ----
    float : 分层估计 - 真 ATE
    '''
    # TODO: 用 make(n=n, seed=seed, uconf=u) 生成数据，
    #       用 strat_ate(..., K) 估计，减去真 ATE
    raise NotImplementedError

In [ ]:
# 自测
print('   u      残余偏差    估计值 / 真 ATE')
_prev = None
for _u in (0.0, 0.25, 0.5, 1.0, 2.0, 4.0):
    _b = residual_bias(_u)
    _ratio = (1.0 + _b) / 1.0
    print(f'  {_u:4.2f}   {_b:+9.4f}      {_ratio:.3f}')
    if _prev is not None:
        assert _b > _prev - 1e-6, f'偏差应随 u 单调不减：u={_u} 时 {_b:.4f} < 前一档 {_prev:.4f}'
    _prev = _b

assert abs(residual_bias(0.0)) < 0.02, 'u=0 时残余偏差应接近 0'
assert residual_bias(1.0) > 0.5, 'u=1 时残余偏差应显著'

# 找出让估计值翻倍的 u（敏感性分析的典型问法）
_us = np.linspace(0, 4, 17)
_bs = np.array([residual_bias(_u) for _u in _us])
_idx = int(np.argmax(_bs >= 1.0))
print()
print(f'✅ 让估计值翻倍（偏差 >= 真 ATE = 1.0）所需的 u ≈ {_us[_idx]:.2f}')
print('   敏感性分析的用法：如果业务上「存在这么强的未观测混杂」不可信，')
print('   结论就相对稳健；如果可信，结论就必须带上这个限定条件。')

## ✏️ 练习 4：定向投放的盈亏平衡

实现 `targeting_breakeven(h)`：给定异质性强度 $h$，用**二分**找出使定向投放
收益恰好等于全量投放（比值 $=1$）的 CATE 估计噪声 $\text{sd}^*$。
超过这个噪声水平，定向就是净亏。

自测会检验一条标度律：$\text{sd}^* \propto \text{sd}(\tau)^2$，
即**异质性翻倍，能容忍的 CATE 噪声涨约 4 倍**。
（用网格搜索会因为分辨率不够而测不出这条律 —— 必须用二分。）

In [ ]:
def targeting_breakeven(h, lo=1e-4, hi=4000.0, iters=40):
    '''二分找出定向投放的盈亏平衡噪声水平 sd*。

    targeting_gain(h, sd)[0] 随 sd 单调递减，所以可以二分：
    维持不变式 gain(lo) >= 1.0 > gain(hi)。

    参数
    ----
    h     : 异质性强度
    lo,hi : 二分区间
    iters : 迭代次数

    返回
    ----
    float : sd*，使 targeting_gain(h, sd*)[0] ≈ 1.0
    '''
    # TODO: 二分。每步取 mid=(lo+hi)/2，若 targeting_gain(h, mid)[0] >= 1.0
    #       则 lo=mid，否则 hi=mid。迭代 iters 次后返回 lo。
    raise NotImplementedError

In [ ]:
# 自测
print('   h    sd(tau)   盈亏平衡 sd*    sd*/sd(tau)   sd*·ATE/sd(tau)²')
_hs = (0.25, 0.5, 0.75, 1.5, 3.0, 6.0, 12.0)
_sd_star, _law = {}, []
for _h in _hs:
    _sd_star[_h] = targeting_breakeven(_h)
    _sdt = targeting_gain(_h, 0.25)[2]
    _ate = targeting_gain(_h, 0.25)[1]
    _law.append(_sd_star[_h] * _ate / _sdt**2)
    print(f'  {_h:5.2f}   {_sdt:6.3f}   {_sd_star[_h]:11.3f}    {_sd_star[_h]/_sdt:9.3f}'
          f'       {_law[-1]:11.3f}')

# ① 单调：h 越大能容忍的噪声越大
for _i in range(1, len(_hs)):
    assert _sd_star[_hs[_i]] > _sd_star[_hs[_i-1]], f'应单调递增：{_sd_star}'

# ② sd*/sd(tau) **不是**常数 —— 先证伪线性猜想
_lin = [_sd_star[_h] / _h for _h in _hs]
assert max(_lin) / min(_lin) > 20, \
    f'若 sd* ∝ sd(tau)，该比值应稳定；实测跨度 {max(_lin)/min(_lin):.1f} 倍 -> 线性猜想被证伪'

# ③ sd*·ATE/sd(tau)² 才是稳定的量（二次律）
assert max(_law) / min(_law) < 1.6, \
    f'sd*·ATE/sd(tau)² 应大致稳定，得到 {[round(v,3) for v in _law]}'

# ④ h 翻倍 -> sd* 涨约 4 倍
for _h in (1.5, 3.0, 6.0):
    _r = _sd_star[_h] / _sd_star[_h / 2]
    assert 3.4 < _r < 4.4, f'h 从 {_h/2} 到 {_h}，sd* 应涨约 4 倍，实测 {_r:.2f}'
    print(f'  h {_h/2:5.2f} -> {_h:5.2f}:  sd* 涨 {_r:.2f} 倍')

print()
print(f'✅ sd*/sd(tau) 跨 {min(_lin):.2f}-{max(_lin):.2f}（{max(_lin)/min(_lin):.0f} 倍跨度）—— 不是线性关系')
print(f'✅ sd*·ATE/sd(tau)² 稳定在 {min(_law):.3f}-{max(_law):.3f} —— **二次**律')
print()
print('   工程含义：能容忍的 CATE 估计噪声 ≈ sd(tau)² / ATE。')
print('   所以判据不是「模型误差小于效应离散度」，而是：')
print('     模型的 CATE 误差 sd 是否小于 sd(tau) × (sd(tau)/ATE)。')
print('   效应离散度只有 ATE 的一半时（sd(tau)/ATE = 0.5），')
print('   容忍的噪声只有 sd(tau) 的一半 —— 定向对模型精度的要求比直觉严得多。')

## 📖 参考答案

In [ ]:
def att_from_counterfactual(Y, T, Y0_treated_mean):
    '''ATT = E[Y|T=1] - E[Y(0)|T=1]。'''
    return float(Y[T == 1].mean() - Y0_treated_mean)

def strat_rmse(K, seeds=20, n=4_000):
    '''K 层分层估计的 RMSE 与被丢弃样本比例（跨 seed）。'''
    errs, drops = [], []
    for s in range(seeds):
        dd = make(n=n, seed=100 + s)
        est, dr = strat_ate(dd['x'], dd['T'], dd['Y'], K, return_dropped=True)
        errs.append(est - dd['tau'].mean())
        drops.append(dr)
    return float(np.sqrt(np.mean(np.square(errs)))), float(np.mean(drops))

def residual_bias(u, n=100_000, K=20, seed=1):
    '''控制全部可观测变量后的残余偏差。'''
    dd = make(n=n, seed=seed, uconf=u)
    return float(strat_ate(dd['x'], dd['T'], dd['Y'], K) - dd['tau'].mean())

def targeting_breakeven(h, lo=1e-4, hi=4000.0, iters=40):
    '''二分找出盈亏平衡噪声水平 sd*。'''
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if targeting_gain(h, mid)[0] >= 1.0:
            lo = mid
        else:
            hi = mid
    return float(lo)

print('参考答案已定义。')
print()
print('要点：')
print('  1. ATT 的两种写法完全等价；把反事实均值换成对照组均值就退化成朴素差，')
print('     两者之差正是选择偏差 —— 所以「估 ATT」= 「估处理组的反事实均值」。')
print('  2. 分层数 K 有内点最优：小 K 偏差主导，大 K 层内样本不足。')
print('     而 K 过大时 RMSE 可能**回落**，因为缺一臂的层被静默跳过，估计目标被换掉了。')
print('  3. 敏感性分析把不可检验的假设变成可讨论的量：')
print('     不问「有没有未观测混杂」，问「要多强才能翻掉结论」。')
print('  4. 定向的盈亏平衡噪声 sd* ∝ sd(tau)^2 / ATE —— 对异质性是**二次**的。')
print('     所以「模型误差小于效应离散度」这条直觉判据太松：真正的判据还要')
print('     再乘一个 sd(tau)/ATE 的因子。')

## 🧪 真实工程胶囊：报告哪一个目标量

下面这段代码把「决策问题 → 目标量」的映射固化成一张表，
并在给定数据上把四个目标量一起算出来，让口径错配无法被忽略。

实际项目里最常见的错配是：**用 ATT 的估计支持 ATC 的决策**。
在这份数据上 ATT = 0.648、ATC = 1.351：
「已经在用的人从中获益 0.65」被当成「推给还没用的人也能获益 0.65」，
而真实值是 1.35（**低估 2.09 倍**）。
错配的方向取决于 $\text{Cov}(\tau, T)$ 的符号，高估和低估都会发生。

In [ ]:
DECISION_TO_TARGET = {
    '要不要全量上线（对全体）':          'ATE',
    '要不要给已经在用的人保留':          'ATT',
    '要不要推给还没用的人':              'ATC',
    '该给哪些人（分人群决策）':          'CATE',
    '要不要对被工具/规则推动的人上线':    'LATE（模块 04）',
}

def report_all_targets(d):
    '''把四个目标量一起算出来，避免口径错配。'''
    T, tau, x = d['T'], d['tau'], d['x']
    out = {
        'ATE': float(tau.mean()),
        'ATT': float(tau[T == 1].mean()),
        'ATC': float(tau[T == 0].mean()),
    }
    print('目标量        值        含义')
    print('-' * 62)
    for k, v in out.items():
        print(f'  {k:6s}   {v:+8.4f}   ' + {
            'ATE': '全人群平均', 'ATT': '已被处理者', 'ATC': '未被处理者'}[k])
    print()
    print('CATE 分位（按 x 十分位）:')
    qs = np.quantile(x, np.linspace(0, 1, 11))
    for i in range(10):
        m = (x >= qs[i]) & (x < qs[i+1] + (1e-9 if i == 9 else 0))
        print(f'  第 {i+1:2d} 十分位  CATE = {tau[m].mean():+8.4f}'
              + ('   <- 效应为负' if tau[m].mean() < 0 else ''))
    out['CATE_min'] = float(min(tau[(x >= qs[i]) & (x < qs[i+1] + 1e-9)].mean()
                                for i in range(10)))
    out['CATE_max'] = float(max(tau[(x >= qs[i]) & (x < qs[i+1] + 1e-9)].mean()
                                for i in range(10)))
    return out

res = report_all_targets(d)
print()
print('决策 -> 目标量 的映射:')
for k, v in DECISION_TO_TARGET.items():
    print(f'  {k:26s} -> {v}')
print()
print(f"最常见的错配：用 ATT {res['ATT']:+.4f} 支持「推给还没用的人」，")
print(f"而该决策的目标量是 ATC {res['ATC']:+.4f} —— 相差 {res['ATC']/res['ATT']:.2f} 倍。")
print('这里 ATT < ATC：自选择进来的人本来就会做得不错，处理对他们的边际价值更小。')
print('所以用 ATT 支持推广决策会**低估**收益 —— 错配的方向取决于 Cov(tau, T) 的符号，')
print('两个方向都会发生，而两者的共同点是：报告的量与决策的量不是同一个。')
print()
print(f"CATE 跨十分位从 {res['CATE_min']:+.4f} 到 {res['CATE_max']:+.4f}，")
print(f"而 ATE 是 {res['ATE']:+.4f}。ATE 是这条曲线的一个加权平均，")
print('不是它的代表值 —— 决策若是分人群的，就必须看曲线。')

assert res['ATC'] > res['ATE'] > res['ATT']
assert res['CATE_min'] < 0 < res['CATE_max']